In [7]:
import joblib
import numpy as np
import pandas as pd
import sqlite3


def load_new_data():
  # Connect to the database and pull a small sample of data to test predictions on
  conn = sqlite3.connect("Regression.db")
  query = "SELECT * FROM electric_motor_temperature LIMIT 5"
  df = pd.read_sql(query, conn)
  conn.close()
  return df


def predict_temperature():
  print("Loading saved model and scaler...")
  try:
    model = joblib.load("pmsm_model.pkl")
    scaler = joblib.load("scaler.pkl")
  except FileNotFoundError as e:
    print(
        f"Error: {e}. Please make sure you run train.py successfully first to"
        " save the model and scaler."
    )
    return

  print("Loading test data for prediction...")
  df = load_new_data()

  # Drop target column if present in the data preview
  if "pm" in df.columns:
    df_features = df.drop(columns=["pm"])
  else:
    df_features = df.copy()

  # Apply the exact same preprocessing and feature engineering as train.py
  if "timestamp" in df_features.columns:
    df_features = df_features.drop(columns=["timestamp"])

  if "u_q" in df_features.columns and "u_d" in df_features.columns:
    df_features["u_magnitude"] = np.sqrt(
        df_features["u_q"] ** 2 + df_features["u_d"] ** 2
    )

  # Scale features using the saved scaler
  X_scaled = pd.DataFrame(
      scaler.transform(df_features), columns=df_features.columns
  )

  print("Generating predictions...")
  predictions = model.predict(X_scaled)

  # Display the results
  print("\n--- Temperature Predictions (Permanent Magnet 'pm') ---")
  for i, pred in enumerate(predictions):
    print(f"Sample {i + 1}: Predicted PM Temperature = {pred:.2f} °C")


if __name__ == "__main__":
  predict_temperature()